~~~
Copyright 2025 Google LLC

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

    https://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.
~~~

# 使用Hugging Face 微調TxGemma

<table><tbody><tr> <td style="text-align: center">    <a href="https://colab.research.google.com/github/google-gemini/gemma-cookbook/blob/main/TxGemma/[TxGemma]Finetune_with_Hugging_Face.ipynb">
      <img alt="Google Colab logo" src="https://www.tensorflow.org/images/colab_logo_32px.png" width="32px"><br> Run in Google Colab
    </a>
</td> <td style="text-align: center">    <a href="https://github.com/google-gemini/gemma-cookbook/blob/main/TxGemma/%5BTxGemma%5DFinetune_with_Hugging_Face.ipynb">
      <img alt="GitHub logo" src="https://github.githubassets.com/assets/GitHub-Mark-ea2971cee799.png" width="32px"><br> View on GitHub
    </a>
</td> <td style="text-align: center">    <a href="https://huggingface.co/collections/google/txgemma-release-67dd92e931c857d15e4d1e87">
      <img alt="Hugging Face logo" src="https://huggingface.co/front/assets/huggingface_logo-noborder.svg" width="32px"><br> View on Hugging Face
    </a>
</td>
</tr></tbody></table>
該notebook 示範了fine-tuning TxGemma 模型，可使用Hugging Face 庫推廣到新的治療開發任務。
此示範使用 Hugging Face 的 [Transformer Reinforcement Learning (`TRL`)](https://github.com/huggingface/trl) library，透過 Supervised Fine-Tuning（SFT）來訓練模型，並結合 [Parameter-Efficient Fine-Tuning (`PEFT`)](https://github.com/huggingface/peft) 與 Low-Rank Adaptation（LoRA）來降低運算成本。訓練資料包含 [TrialBench](https://arxiv.org/abs/2407.00631) dataset 的子集，用於微調 TxGemma，以預測臨床試驗中的不良事件。

## 設定

要完成本教學，您需要擁有 Colab runtime 以及足夠的資源來微調和執行 TxGemma 模型。在這種情況下，您可以使用 T4 GPU：
1. 在 Colab 視窗的右上角，選擇 **▾（其他連接選項）**。
2. 選擇**更改 runtime 類型**。
3. 在 **硬體加速器** 下，選擇 **T4 GPU**。

### 訪問TxGemma

在開始之前，請確保您有權訪問 Hugging Face 上的 TxGemma 模型：
1. 如果您還沒有Hugging Face 帳戶，您可以點選[此處](https://huggingface.co/join) 免費建立帳戶。
2. 前往 [TxGemma 型號頁面](https://huggingface.co/google/txgemma-2b-predict) 並接受使用條件。

### 設定您的 HF token

點選[此處](https://huggingface.co/settings/tokens)產生Hugging Face `read`存取token並將您的存取token新增至ColabSecrets manager 以安全地儲存它。
1. 開啟 Google Colab notebook 並點選左側面板中的 🔑 Secrets 標籤。 <img src="https://storage.googleapis.com/generativeai-downloads/images/secrets.jpg" alt="The Secrets tab is found on the left panel." width=50%>
2. 建立一個新的secret，名稱為`HF_TOKEN`。
3. 將token 金鑰複製/貼上到`HF_TOKEN` 的值輸入框中。
4. 切換左側的按鈕以允許notebook 存取secret。

In [ ]:
import os
from google.colab import userdata
# Note: `userdata.get` is a Colab API. If you're not using Colab, set the env
# vars as appropriate for your system.
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

### 安裝依賴項

In [ ]:
! pip install --upgrade --quiet bitsandbytes datasets peft transformers trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.0/76.0 MB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 411.0/411.0 kB 38.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 110.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.4/336.4 kB 29.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 57.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 36.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 54.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## 從Hugging Face Hub載入模型

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_id = "google/txgemma-2b-predict"

# Use 4-bit quantization to reduce memory usage
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    device_map={"":0},
    torch_dtype="auto",
    attn_implementation="eager",
)

tokenizer_config.json:   0%|          | 0.00/46.4k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/818 [00:00<?, ?B/s]

...

model-00003-of-00003.safetensors:   0%|          | 0.00/481M [00:00<?, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

## 加載dataset

此notebook使用來自[TrialBench](https://arxiv.org/abs/2407.00631)的不良事件預測資料來微調TxGemma。 dataset已預處理為指令調優格式，並可在[雲端儲存](https://console.cloud.google.com/storage/browser/healthai-us/txgemma/datasets)中使用。
使用Hugging Face [`datasets`](https://github.com/huggingface/datasets) library 載入dataset。

In [ ]:
from datasets import load_dataset

! wget -nc https://storage.googleapis.com/healthai-us/txgemma/datasets/trialbench_adverse-event-rate-prediction_train.jsonl
data = load_dataset(
    "json",
    data_files="/content/trialbench_adverse-event-rate-prediction_train.jsonl",
    split="train",
)

# Display dataset details
data

--2025-04-06 06:36:33--  https://storage.googleapis.com/healthai-us/txgemma/datasets/trialbench_adverse-event-rate-prediction_train.jsonl
Resolving storage.googleapis.com (storage.googleapis.com)... 64.233.180.207, 142.251.167.207, 172.253.115.207, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|64.233.180.207|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 18413655 (18M) [application/octet-stream]
Saving to: ‘trialbench_adverse-event-rate-prediction_train.jsonl’

trialbench_adverse- 100%[===================>]  17.56M  63.5MB/s    in 0.3s    

2025-04-06 06:36:34 (63.5 MB/s) - ‘trialbench_adverse-event-rate-prediction_train.jsonl’ saved [18413655/18413655]



Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['input_text', 'output_text'],
    num_rows: 14368
})

每個數據點包括：
* `"input_text"`：問題，prompt是根據臨床試驗資訊預測是否會出現不良事件的模型。輸入包括藥物 SMILES 字串和文字訊息。

* `"output_text"`：回答，要么是“是”，要么是“否”。

以下是 dataset 的範例：

In [ ]:
data["input_text"][0]

'From the following information about a clinical trial, predict whether it would have an adverse event.\n\nTitle: Safety, Tolerability and Pharmacokinetics of Single and Repeat Doses of GSK2292767 in Healthy Participants Who Smoke Cigarettes\nSummary: This study is the first administration of GSK2292767 to humans. The study will evaluate the safety, tolerability, pharmacokinetics (PK) and pharmacodynamics (PD) of single and repeat inhaled doses of GSK2292767 in healthy smokers. This study is intended to provide sufficient confidence in the safety of the molecule and preliminary information on target engagement to allow progression to further repeat dose and proof of mechanism studies. This is a two part, single site, randomized, double-blind (sponsor open), placebo controlled study. Part A will consist of two 3-period interlocking cohorts to evaluate the safety, tolerability and pharmacokinetics of ascending single doses of GSK2292767 administered as a dry powder inhalation. Part B is 

In [ ]:
data["output_text"][0]

'No'

訓練的預期資料格式是包含完整文字序列的單一 `"text"` 列。
在這裡，定義一個函數來正確格式化 dataset 中的每個範例。在後面的部分中，它將被傳遞到`SFTTrainer`，token化之前將格式化函數應用於dataset。

In [ ]:
def formatting_func(example):
    text = f"{example['input_text']} {example['output_text']}<eos>"
    return text

# Display formatted training data example
print(formatting_func(data[0]))

From the following information about a clinical trial, predict whether it would have an adverse event.

Title: Safety, Tolerability and Pharmacokinetics of Single and Repeat Doses of GSK2292767 in Healthy Participants Who Smoke Cigarettes
Summary: This study is the first administration of GSK2292767 to humans. The study will evaluate the safety, tolerability, pharmacokinetics (PK) and pharmacodynamics (PD) of single and repeat inhaled doses of GSK2292767 in healthy smokers. This study is intended to provide sufficient confidence in the safety of the molecule and preliminary information on target engagement to allow progression to further repeat dose and proof of mechanism studies. This is a two part, single site, randomized, double-blind (sponsor open), placebo controlled study. Part A will consist of two 3-period interlocking cohorts to evaluate the safety, tolerability and pharmacokinetics of ascending single doses of GSK2292767 administered as a dry powder inhalation. Part B is plan

## 嘗試預訓練模型

提示預訓練模型以查看其在範例不良事件預測任務中的執行情況。在 fine-tuning 之前，模型無法理解指令並提供不適當的答案。

In [ ]:
prompt = "From the following information about a clinical trial, predict whether it would have an adverse event.\n\nDrug: C[C@H]1OC2=C(N)N=CC(=C2)C2=C(C#N)N(C)N=C2CN(C)C(=O)C2=C1C=C(F)C=C2\n\nAnswer:"
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens=8)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

From the following information about a clinical trial, predict whether it would have an adverse event.

Drug: C[C@H]1OC2=C(N)N=CC(=C2)C2=C(C#N)N(C)N=C2CN(C)C(=O)C2=C1C=C(F)C=C2

Answer:188


## 使用LoRA微調模型

傳統的大型語言模型 fine-tuning 是資源密集的，因為它需要調整數十億個參數。參數高效微調 (PEFT) 透過使用低階適應 (LoRA) 等技術訓練較少數量的參數來解決這個問題。 LoRA 透過訓練添加到原始模型的小型低秩矩陣而不是更新全權重矩陣，有效地適應大型語言模型。

本節示範 fine-tuning TxGemma 使用 LoRA 和 Hugging Face `TRL` library 中的 `SFTTrainer`。

首先，定義 [`LoraConfig`](https://huggingface.co/docs/peft/main/en/package_reference/lora)，包括適應矩陣的等級和要新增 LoRA 適配器的模型層。

In [ ]:
from peft import LoraConfig

lora_config = LoraConfig(
    r=8,
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "o_proj",
        "k_proj",
        "v_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

準備訓練模型。

In [ ]:
from peft import prepare_model_for_kbit_training, get_peft_model

# Preprocess quantized model for training
model = prepare_model_for_kbit_training(model)

# Create PeftModel from quantized model and configuration
model = get_peft_model(model, lora_config)

此範例使用監督微調 (SFT) 方法來訓練 TxGemma 模型。
在這裡，建立處理完整訓練循環的 [`SFTTrainer`](https://huggingface.co/docs/trl/sft_trainer)，包括資料載入、前向和後向傳遞以及優化器步驟。指定先前定義的LoRA設定和dataset格式化函數以及帶有訓練參數的`SFTConfig`。

In [ ]:
import transformers
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    train_dataset=data,
    args=SFTConfig(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        warmup_steps=2,
        max_steps=50,
        learning_rate=2e-4,
        fp16=True,
        logging_steps=5,
        max_seq_length=512,
        output_dir="/content/outputs",
        optim="paged_adamw_8bit",
        report_to="none",
    ),
    peft_config=lora_config,
    formatting_func=formatting_func,
)

Applying formatting function to train dataset:   0%|          | 0/14368 [00:00<?, ? examples/s]

Converting train dataset to ChatML:   0%|          | 0/14368 [00:00<?, ? examples/s]

Applying chat template to train dataset:   0%|          | 0/14368 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/14368 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/14368 [00:00<?, ? examples/s]

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


啟動 fine-tuning 進程。

In [ ]:
trainer.train()

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
5,16.750800
10,10.910500
15,8.549700
20,6.355500
25,5.270800
30,4.851800
35,4.085300
40,3.853700
45,3.727900
50,3.595500


TrainOutput(global_step=50, training_loss=6.79513858795166, metrics={'train_runtime': 158.6588, 'train_samples_per_second': 1.261, 'train_steps_per_second': 0.315, 'total_flos': 726227766793728.0, 'train_loss': 6.79513858795166})

## 測試微調後的模型

提示微調模型以查看其在範例不良事件預測任務中的執行情況。在fine-tuning之後，模型學會了用適當的答案來回應prompt。

In [ ]:
prompt = "From the following information about a clinical trial, predict whether it would have an adverse event.\n\nDrug: C[C@H]1OC2=C(N)N=CC(=C2)C2=C(C#N)N(C)N=C2CN(C)C(=O)C2=C1C=C(F)C=C2\n\nAnswer:"
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens=8)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/torch/utils/checkpoint.py:87: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


From the following information about a clinical trial, predict whether it would have an adverse event.

Drug: C[C@H]1OC2=C(N)N=CC(=C2)C2=C(C#N)N(C)N=C2CN(C)C(=O)C2=C1C=C(F)C=C2

Answer: Yes


# 後續步驟

探索其他 [notebooks](https://github.com/google-gemini/gemma-cookbook/blob/main/TxGemma) 以了解您還可以使用該模型做什麼。